# 06 — ML Meta-Labeling
The strategy proposes every candidate trade; a LightGBM classifier learns
which entry conditions historically won, under purged + embargoed
time-series CV. SHAP explains what the model uses. The score filter is then
compared against the unfiltered baseline **on out-of-fold trades only**.

In [ ]:
import sys
sys.path.insert(0, "../src")
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (11, 4)
pd.set_option("display.width", 160)

In [ ]:
from lab.backtest import StrategyConfig
from lab.ml import build_dataset, train_metalabel, shap_importance, leakage_check

cfg = StrategyConfig.from_yaml("../configs/short_put_45dte.yaml").replace(
    start="2010-01-01", end="2023-12-31")
ds = build_dataset(cfg)
print(f"{len(ds):,} candidate trades, {ds.entry_date.min():%Y-%m} to {ds.entry_date.max():%Y-%m}")
ds.head()

In [ ]:
res = train_metalabel(ds, n_splits=5, embargo_days=5)
res.fold_metrics.round(3)

A mean out-of-fold AUC meaningfully above 0.5 (and top-half win rate above
base) is required before the filter deserves any capital.

In [ ]:
auc = res.fold_metrics.auc.mean()
print(f"mean OOF AUC: {auc:.3f}")
print(f"leakage check (shuffled labels, must be ~0.5): {leakage_check(ds):.3f}")

In [ ]:
shap_importance(res);

## Score filter vs baseline (out-of-fold trades only)

In [ ]:
for thr in (0.5, 0.6, 0.7):
    print(f"--- threshold {thr}")
    print(res.apply_filter(thr).round(3), chr(10))

In [ ]:
# Equity comparison on OOF trades, averaged per entry date
oof = res.dataset.dropna(subset=["score"])
daily_base = oof.groupby("entry_date")["pct_change"].mean().cumsum()
picked = oof[oof.score >= 0.6]
daily_ml = picked.groupby("entry_date")["pct_change"].mean().cumsum()
ax = daily_base.plot(label="all trades")
daily_ml.plot(ax=ax, label="score >= 0.6")
ax.legend(); ax.set_title("Cumulative avg trade return: baseline vs ML-filtered");